In [6]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [7]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [8]:
Path("../outputs/metrics").mkdir(parents=True, exist_ok=True)
Path("../outputs/models").mkdir(parents=True, exist_ok=True)

In [9]:
with open("../data/graph/train_graph.pkl", "rb") as f:
    train_graph = pickle.load(f)

with open("../data/graph/test_graph.pkl", "rb") as f:
    test_graph = pickle.load(f)

with open("../data/graph/wallet_mapping.pkl", "rb") as f:
    wallet_mapping = pickle.load(f)

num_nodes = wallet_mapping["num_nodes"]

print("Num nodes:", num_nodes)
print("Train edges:", train_graph["edge_index"].shape)
print("Test edges:", test_graph["edge_index"].shape)

Num nodes: 108582
Train edges: (2, 38454)
Test edges: (2, 298160)


In [10]:
class EdgeDataset(Dataset):
    def __init__(self, graph):
        self.edge_index = torch.tensor(graph["edge_index"], dtype=torch.long)
        self.edge_features = torch.tensor(graph["edge_features"], dtype=torch.float32)
        self.labels = torch.tensor(graph["edge_labels"], dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        source = self.edge_index[0, idx]
        target = self.edge_index[1, idx]
        edge_feat = self.edge_features[idx]
        label = self.labels[idx]

        return source, target, edge_feat, label

In [11]:
BATCH_SIZE = 1024

train_dataset = EdgeDataset(train_graph)
test_dataset = EdgeDataset(test_graph)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [12]:
class GraphEdgeClassifier(nn.Module):
    def __init__(self, num_nodes, edge_feat_dim, embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.node_embedding = nn.Embedding(num_nodes, embedding_dim)

        input_dim = embedding_dim * 2 + edge_feat_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        tgt_emb = self.node_embedding(target)

        x = torch.cat([src_emb, tgt_emb, edge_feat], dim=1)

        logits = self.mlp(x).squeeze(1)

        return logits

In [13]:
edge_feat_dim = train_graph["edge_features"].shape[1]

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128
).to(DEVICE)

model

GraphEdgeClassifier(
  (node_embedding): Embedding(108582, 64)
  (mlp): Sequential(
    (0): Linear(in_features=138, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [14]:
labels = train_graph["edge_labels"]

num_positive = np.sum(labels == 1)
num_negative = np.sum(labels == 0)

pos_weight_value = num_negative / num_positive

print("Positive:", num_positive)
print("Negative:", num_negative)
print("pos_weight:", pos_weight_value)

Positive: 754
Negative: 37700
pos_weight: 50.0


In [15]:
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [17]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()

        logits = model(source, target, edge_feat)
        loss = criterion(logits, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [18]:
def evaluate(model, loader, threshold=0.3):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for source, target, edge_feat, label in loader:
            source = source.to(DEVICE)
            target = target.to(DEVICE)
            edge_feat = edge_feat.to(DEVICE)

            logits = model(
                source,
                target,
                edge_feat
            )

            probs = torch.sigmoid(logits)

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                label.numpy()
            )

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    preds = (all_probs >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "confusion_matrix": confusion_matrix(all_labels, preds)
    }

    return metrics

In [19]:
EPOCHS = 20

history = []

best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_metrics = evaluate(model, test_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "roc_auc": test_metrics["roc_auc"],
        "pr_auc": test_metrics["pr_auc"]
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {test_metrics['accuracy']:.4f} | "
        f"Prec: {test_metrics['precision']:.4f} | "
        f"Rec: {test_metrics['recall']:.4f} | "
        f"F1: {test_metrics['f1']:.4f} | "
        f"ROC-AUC: {test_metrics['roc_auc']:.4f} | "
        f"PR-AUC: {test_metrics['pr_auc']:.4f}"
    )

    if test_metrics["f1"] > best_f1:
        best_f1 = test_metrics["f1"]

        torch.save(
            model.state_dict(),
            "../outputs/models/graph_class_weight_best.pt"
        )

Epoch 01 | Loss: 1.2563 | Acc: 0.0305 | Prec: 0.0002 | Rec: 1.0000 | F1: 0.0004 | ROC-AUC: 0.6708 | PR-AUC: 0.0021
Epoch 02 | Loss: 1.0753 | Acc: 0.2030 | Prec: 0.0002 | Rec: 0.9062 | F1: 0.0005 | ROC-AUC: 0.6942 | PR-AUC: 0.0016
Epoch 03 | Loss: 0.9430 | Acc: 0.5137 | Prec: 0.0003 | Rec: 0.7188 | F1: 0.0006 | ROC-AUC: 0.6943 | PR-AUC: 0.0017
Epoch 04 | Loss: 0.8115 | Acc: 0.5222 | Prec: 0.0003 | Rec: 0.7500 | F1: 0.0007 | ROC-AUC: 0.7034 | PR-AUC: 0.0021
Epoch 05 | Loss: 0.6846 | Acc: 0.6041 | Prec: 0.0004 | Rec: 0.6719 | F1: 0.0007 | ROC-AUC: 0.6927 | PR-AUC: 0.0056
Epoch 06 | Loss: 0.5684 | Acc: 0.7349 | Prec: 0.0004 | Rec: 0.5156 | F1: 0.0008 | ROC-AUC: 0.6977 | PR-AUC: 0.0193
Epoch 07 | Loss: 0.4367 | Acc: 0.7422 | Prec: 0.0004 | Rec: 0.4688 | F1: 0.0008 | ROC-AUC: 0.6876 | PR-AUC: 0.0700
Epoch 08 | Loss: 0.3324 | Acc: 0.8185 | Prec: 0.0004 | Rec: 0.3750 | F1: 0.0009 | ROC-AUC: 0.6946 | PR-AUC: 0.0707
Epoch 09 | Loss: 0.2665 | Acc: 0.8556 | Prec: 0.0006 | Rec: 0.3906 | F1: 0.0012 

In [20]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "../outputs/metrics/graph_class_weight_history.csv",
    index=False
)

history_df

,epoch,train_loss,accuracy,precision,recall,f1,roc_auc,pr_auc
0,1,1.256321,0.030537,0.000221,1.000000,0.000443,0.670796,0.002097
1,2,1.075297,0.202998,0.000244,0.906250,0.000488,0.694215,0.001593
2,3,0.942998,0.513701,0.000317,0.718750,0.000634,0.694326,0.001741
3,4,0.811468,0.522193,0.000337,0.750000,0.000673,0.703446,0.002055
4,5,0.684591,0.604062,0.000364,0.671875,0.000728,0.692721,0.005636
5,6,0.568398,0.734901,0.000417,0.515625,0.000834,0.697732,0.019251
6,7,0.436654,0.742195,0.000390,0.468750,0.000780,0.687641,0.070002
7,8,0.332437,0.818507,0.000444,0.375000,0.000886,0.694604,0.070728
8,9,0.266515,0.855624,0.000581,0.390625,0.001160,0.681717,0.066606
9,10,0.213268,0.864217,0.000568,0.359375,0.001135,0.689788,0.066495


In [21]:
model.load_state_dict(
    torch.load("../outputs/models/graph_class_weight_best.pt")
)

final_metrics = evaluate(model, test_loader)

print("Final Metrics:")
for k, v in final_metrics.items():
    if k != "confusion_matrix":
        print(k, ":", v)

print("Confusion Matrix:")
print(final_metrics["confusion_matrix"])

Final Metrics:
threshold : 0.3
accuracy : 0.9586698416957339
precision : 0.001220603792009114
recall : 0.234375
f1 : 0.0024285598640006478
roc_auc : 0.6921787570111643
pr_auc : 0.06545035915376227
Confusion Matrix:
[[285822  12274]
 [    49     15]]


In [22]:
final_result = {
    "model": "graph_class_weight",
    "accuracy": final_metrics["accuracy"],
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "f1": final_metrics["f1"],
    "roc_auc": final_metrics["roc_auc"],
    "pr_auc": final_metrics["pr_auc"],
    "tn": final_metrics["confusion_matrix"][0, 0],
    "fp": final_metrics["confusion_matrix"][0, 1],
    "fn": final_metrics["confusion_matrix"][1, 0],
    "tp": final_metrics["confusion_matrix"][1, 1],
}

pd.DataFrame([final_result]).to_csv(
    "../outputs/metrics/graph_class_weight_final.csv",
    index=False
)

final_result

{'model': 'graph_class_weight',
 'accuracy': 0.9586698416957339,
 'precision': 0.001220603792009114,
 'recall': 0.234375,
 'f1': 0.0024285598640006478,
 'roc_auc': 0.6921787570111643,
 'pr_auc': 0.06545035915376227,
 'tn': np.int64(285822),
 'fp': np.int64(12274),
 'fn': np.int64(49),
 'tp': np.int64(15)}